In [15]:
import os
from pathlib import Path
from typing import Literal, Optional, List
import yaml

import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import awkward as ak
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec
import seaborn as sns

os.environ["PROJECT_PATH"] = "/home/syebes/lambda-ana"
project_path = Path(os.environ["PROJECT_PATH"])
sys.path.append(str(project_path))

# My imports
from src.driver import AnalysisDriver
import src.branches as branches
from src.cuts import CutConfig, create_cut

# Load configurations

In [16]:
with open("../../config/cuts_config.yaml", "r") as f:
    config = yaml.safe_load(f)

cuts_config = []

for cut_cfg in config['frans_cuts'].values():
    cfg = CutConfig(
        cut = create_cut(cut_cfg['cut']),
        **{k: v for k, v in cut_cfg.items() if k != "cut"}
    )

    cuts_config.append(cfg)

# Load files

In [2]:
# POT taken from preselection stage
aurora_pot = 5.401777280854836e+20

aurora_file = "/exp/sbnd/data/users/syebes/eafData/fransOutputs/devFransOutput.root"
frans_tdir = "FRANSProducer/fransTree"

aurora_tree = uproot.open(aurora_file)[frans_tdir]

In [3]:
cut_list = [
    "pre_pass",
    "frans_n_angles >= 1",
    "frans_score >= 0.2",
    #"michel_final_score >= 0.9",
    "frans_contained_phi < 1.8",
    "frans_pzsign == 2",
    "frans_n_unassoc_origins < 1",
    "frans_n_max_dirt_unassoc_hits < 10",
    #"pid_cut == 1",
    #"InvariantMass > 1000",
    #"InvariantMass < 1500",
]

tree_cuts = ""

for c in cut_list:
    if not tree_cuts:
        tree_cuts += c
    else:    
        tree_cuts = tree_cuts + " & " + c

In [4]:
tree_cuts = (
    "(pre_pass) & "
    "(frans_n_angles >= 1) & "
    "(frans_score >= 0.2) & "
    "(frans_contained_phi < 1.8) & "
    "(frans_pzsign == 2) & "
    "(frans_n_unassoc_origins < 1) & "
    "(frans_n_max_dirt_unassoc_hits < 10)"
)

In [5]:
max_entry = None
indices =['run_id', 'subrun_id', 'event_id', 'slice_id']

arr = aurora_tree.arrays(branches.frans_and_true, 
                                     library="ak", 
                                     cut = tree_cuts,
                                     entry_stop=max_entry)

In [6]:
ak.to_parquet(arr, "/exp/sbnd/data/users/syebes/eafData/true_aurora_frans_arr.parquet")

  created_by: parquet-cpp-arrow version 24.0.0
  num_columns: 113
  num_rows: 64
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 0

# Run remaining cuts

## FRANS cuts

In [17]:
arr['michel_final_score'] = ak.where(~arr.michel_vars_computed, 2, arr.michel_score)

In [21]:
# Michel score cut

michel_mask = (arr.michel_final_score > 0.9)

In [22]:
michel_mask

<Array [False, True, True, True, ..., True, False, False] type='64 * bool'>

## PID cuts

In [ ]:
pid_cuts_config = []

for cut_cfg in config['pid_cuts'].values():
    cfg = CutConfig(
        cut = create_cut(cut_cfg['cut']),
        **{k: v for k, v in cut_cfg.items() if k != "cut"}
    )

    pid_cuts_config.append(cfg)

In [58]:
selected_indices = np.array(ak.zip([
    arr[michel_mask].run_id,
    arr[michel_mask].subrun_id,
    arr[michel_mask].event_id,
]).to_list())

In [49]:
pid_tdir = "lambdaPidPandoraAna/CalorimetryTree"

aurora_pid_tree = uproot.open(aurora_file)[pid_tdir]
pid_df = pd.DataFrame(aurora_pid_tree.arrays(branches.frans_pid, library="np", entry_stop=max_entry))

pid_df.set_index(['RunID', 'SubRunID', 'EventID'], inplace=True)

In [62]:
pid_df = pid_df[pid_df.index.isin(list(selected_indices))]

cols = ['is_lambda_signal', 'scale', 'gen_index', 'n_hyperon']

pid_df['pid_cut'] = (pid_df.NTracksHI >=1) & (pid_df.NTracksLI >=1)

In [63]:
pid_cut_mask = (pid_df.pid_cut) & (pid_df.InvariantMass>1000) & (pid_df.InvariantMass<1500)

In [64]:
final_indices = np.array(pid_df[pid_cut_mask].index.to_list())

In [72]:
pid_ak_mask = np.isin(selected_indices, final_indices).all(axis=1)

In [73]:
pid_ak_mask

array([False, False,  True,  True,  True,  True, False, False,  True,
       False,  True,  True,  True, False,  True,  True,  True,  True,
        True, False, False, False,  True,  True, False, False, False,
        True, False, False, False,  True,  True, False,  True, False])

In [74]:
final_arr = arr[michel_mask][pid_ak_mask]

In [76]:
len(final_arr)

19

# Classify backgrounds

In [80]:
n_signal = sum(final_arr.is_lambda_signal)
n_bck = sum(~final_arr.is_lambda_signal)

print(f"Signal: {n_signal}")
print(f"Background: {n_bck}")

Signal: 10
Background: 9


In [82]:
final_arr[~final_arr.is_lambda_signal].n_hyperon

<Array [0, 0, 0, 0, 0, 0, 0, 0, 1] type='9 * int32'>

In [83]:
ak.to_parquet(final_arr, "/exp/sbnd/data/users/syebes/eafData/true_aurora_selected.parquet")

  created_by: parquet-cpp-arrow version 24.0.0
  num_columns: 114
  num_rows: 19
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 0

In [86]:
bck_arr = final_arr[~final_arr.is_lambda_signal & (final_arr.n_hyperon==0)]

In [89]:
gen_stable = (bck_arr.gen_status_code == 1)

bck_arr.gen_pdg[gen_stable]

<Array [[13, 2212], [...], ..., [22, 13, 2112, 2212]] type='8 * var * int32'>

In [94]:
bck_arr.gen_pdg[gen_stable][~ak.any(bck_arr.gen_pdg[gen_stable] == 2112, axis=1)]

<Array [[13, 2212], [13, ...], [13, 2212, 211]] type='3 * var * int32'>

In [109]:
neutron_events = bck_arr[ak.any(bck_arr.gen_pdg[gen_stable] == 2112, axis=1)]

In [110]:
g4_prim_neutrons = (neutron_events.g4_pdg == 2112) & (neutron_events.g4_process == "primary")

In [129]:
neutron_events.g4_tl[g4_prim_neutrons]

<Array [[6.81], [267], ..., [23.6, ...], [6.91]] type='5 * var * float64'>

In [117]:
neutron_events.g4_mother

<Array [[10008245, 10008245, ..., 10000000], ..., [...]] type='5 * var * int32'>

In [127]:
neutron_dau_mask = ak.any(
    neutron_events.g4_mother[:, :, None] == neutron_events.g4_id[g4_prim_neutrons][:, None, :],
    axis=-1
) & (neutron_events.g4_pdg < 10000) & (neutron_events.g4_tl > 3) 

In [128]:
neutron_events.g4_pdg[neutron_dau_mask]

<Array [[22, 2112, 2112, ..., 2112, 2212, 2212], ...] type='5 * var * int32'>

In [126]:
neutron_events.g4_tl[neutron_dau_mask]

<Array [[24.8, 2.66e+03, 63.2, ..., 4.44, 7.48], ...] type='5 * var * float64'>

In [130]:
no_neutron_events = neutron_events = bck_arr[~ak.any(bck_arr.gen_pdg[gen_stable] == 2112, axis=1)]

In [138]:
no_neutron_events[['run_id','subrun_id','event_id']]

<Array [{run_id: 1181, ...}, {...}, {...}] type='3 * {run_id: uint32, subru...'>

In [88]:
bck_arr.fields

['file_name',
 'event_id',
 'run_id',
 'subrun_id',
 'slice_id',
 'gen_index',
 'nu_pdg',
 'nu_int_mode',
 'nu_int_type',
 'nu_cc_nc',
 'gen_id',
 'gen_status_code',
 'gen_mother',
 'gen_pdg',
 'gen_mass',
 'gen_e0',
 'gen_start_x',
 'gen_start_y',
 'gen_start_z',
 'gen_p0_x',
 'gen_p0_y',
 'gen_p0_z',
 'n_protons',
 'n_neutrons',
 'n_pi0',
 'n_pip',
 'n_pim',
 'n_mup',
 'n_mum',
 'n_ep',
 'n_em',
 'n_lambda',
 'n_sigma0',
 'n_sigmap',
 'n_sigmam',
 'n_hyperon',
 'g4_id',
 'g4_mother',
 'g4_pdg',
 'g4_e0',
 'g4_process',
 'g4_end_process',
 'g4_mass',
 'g4_ef',
 'g4_tl',
 'true_gap',
 'has_lambda',
 'has_lambda_v',
 'is_lambda_signal',
 'slc_purity',
 'slc_completeness',
 'pre_is_clear_cosmic',
 'pre_nu_score',
 'pre_bcfm_score',
 'pre_is_fiducial',
 'pre_n_prim_pfps',
 'pre_n_prim_trks',
 'pre_n_prim_shwrs',
 'pre_n_trks_exit',
 'pre_max_gap',
 'pre_pass',
 'frans_n_origins',
 'frans_n_origins_mult1',
 'frans_n_origins_mult2',
 'frans_n_origins_gt3',
 'frans_n_origins_pair_one_two',
 